# Phase 4: RAG Generation
This notebook implements context-grounded generation using Qwen2.5-7B-Instruct (4-bit) and the retrieval pipeline from Phase 3.


### Module 1: Setup & Dependencies
Installs all required libraries, mounts Google Drive, and centralizes configuration constants.


In [ ]:
!pip uninstall -y transformers gptqmodel
!pip install -q sentence-transformers rank_bm25 networkx "numpy<2.0.0" tqdm "transformers==4.45.0" accelerate

# Install the complex packages one by one so we can see exactly which one breaks
!pip install autoawq
!pip install gptqmodel

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# ================================================================
# CONFIG
# ================================================================
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S", stream=sys.stdout)
log = logging.getLogger("tiet_rag_generation")

CHECKPOINT_DIR = "/content/drive/MyDrive/tiet_rag2/knowledge_graph"
QA_DATASET_DIR = "/content/drive/MyDrive/tiet_rag2/qa_dataset"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_CACHE_DIR = "/content/drive/MyDrive/tiet_rag2/models/qwen2.5-7b-instruct-awq"
EMBEDDER_MODEL = "BAAI/bge-m3"
MAX_CONTEXT_TOKENS = 3500
GENERATION_MAX_NEW_TOKENS = 512

print("✓ Config loaded.")




### Module 2: Load Retrieval Artifacts & Rebuild retrieve()
Loads the registry, embeddings, graph, BM25 index, and embeds the Phase 3 search functions.


In [ ]:
# MODULE 1a: Load chunk_registry and chunk_embeddings
# ================================================================
import json
import os
import numpy as np

# --- Load chunk registry ---
CHUNK_REGISTRY_PATH = f"{CHECKPOINT_DIR}/chunk_registry_full.json"

with open(CHUNK_REGISTRY_PATH) as f:
    chunk_registry = json.load(f)

log.info("Loaded chunk registry: %d chunks", len(chunk_registry))

# --- Load chunk embeddings ---
EMBED_CACHE = f"{CHECKPOINT_DIR}/chunk_embeddings.npy"

chunk_embeddings = np.load(EMBED_CACHE)
log.info("Loaded embeddings: shape %s, dtype %s", chunk_embeddings.shape, chunk_embeddings.dtype)

# --- Validate alignment ---
assert len(chunk_registry) == chunk_embeddings.shape[0], (
    f"MISMATCH: chunk_registry has {len(chunk_registry)} entries "
    f"but embeddings have {chunk_embeddings.shape[0]} rows!"
)
print(f"\u2713 chunk_registry ({len(chunk_registry)}) and embeddings ({chunk_embeddings.shape}) are aligned.")

# --- Verify L2 normalization (spot-check) ---
sample_norms = np.linalg.norm(chunk_embeddings[:10], axis=1)
assert np.allclose(sample_norms, 1.0, atol=1e-3), (
    f"Embeddings are NOT L2-normalized! Sample norms: {sample_norms}"
)
print(f"\u2713 Embeddings are L2-normalized (sample norms: {sample_norms[:3].round(6)})")

# --- Build chunk_id -> index map for fast lookup ---
chunk_id_to_idx = {c["chunk_id"]: i for i, c in enumerate(chunk_registry)}
print(f"\u2713 Built chunk_id_to_idx map ({len(chunk_id_to_idx)} entries)")

# ================================================================
# MODULE 1b: Load knowledge graph from graph_checkpoint.json
#
# IMPORTANT: Load from graph_checkpoint.json, NOT from .gexf/.graphml
# exports — those collapse MultiDiGraph multi-edge keys and are for
# visualization only.
# ================================================================
import networkx as nx

GRAPH_CKPT = f"{CHECKPOINT_DIR}/graph_checkpoint.json"

with open(GRAPH_CKPT) as f:
    ckpt = json.load(f)

# --- Rebuild MultiDiGraph ---
G = nx.MultiDiGraph()

for node_id, node_data in ckpt.get("nodes", {}).items():
    G.add_node(node_id, **node_data)

for edge in ckpt.get("edges", []):
    edge_data = dict(edge)
    key = edge_data.pop("key", edge_data.get("relation"))
    src = edge_data.pop("source")
    tgt = edge_data.pop("target")
    G.add_edge(src, tgt, key=key, **edge_data)

# --- Rebuild name_index and alias_index ---
name_index = {}   # canonical_name -> node_id
alias_index = {}  # alias (any form) -> node_id

for n, d in G.nodes(data=True):
    if "canonical_name" in d:
        name_index[d["canonical_name"]] = n
    for al in d.get("aliases", []):
        alias_index[al] = n
        alias_index[al.lower().strip()] = n

log.info(
    "Graph loaded: %d nodes, %d edges | name_index: %d | alias_index: %d",
    G.number_of_nodes(), G.number_of_edges(),
    len(name_index), len(alias_index),
)
print(f"\u2713 Knowledge graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\u2713 name_index: {len(name_index)} entries, alias_index: {len(alias_index)} entries")

# --- Show a few sample nodes ---
print("\nSample nodes:")
for nid, ndata in list(G.nodes(data=True))[:5]:
    print(f"  {nid}  (type={ndata.get('type')}, sources={len(ndata.get('source_chunks', []))})")

# ================================================================
# MODULE 1c: Load QA dataset (for sanity testing in Module 7)
#
# ⚠️  EVAL LEAKAGE WARNING:
# This QA dataset is graph-derived ground truth. It is fine for
# sanity-checking retrieval (Module 7), but must NOT be used as
# the final evaluation set after the retrieval system is tuned
# against it — that risks overfitting. A separate held-out set
# should be generated in Phase 5 for final reported metrics.
# ================================================================
import glob

qa_files = sorted(glob.glob(f"{QA_DATASET_DIR}/tiet_rag_dataset_*.json"))
if not qa_files:
    print("\u26a0 No QA dataset found — Module 7 sanity tests will be skipped.")
    qa_dataset = []
else:
    qa_path = qa_files[-1]  # use the latest file
    with open(qa_path) as f:
        qa_dataset = json.load(f)
    log.info("Loaded QA dataset from %s: %d questions", os.path.basename(qa_path), len(qa_dataset))

    # --- Distribution by hop type ---
    from collections import Counter
    hop_dist = Counter(q.get("hop_type", "unknown") for q in qa_dataset)
    print(f"\u2713 QA dataset: {len(qa_dataset)} questions")
    print(f"  Hop-type distribution: {dict(hop_dist)}")

# ================================================================
# MODULE 1d: Load embedding model (for query encoding)
#
# Note: Do NOT load the Qwen LLM here — defer until the generation
# step at the very end. This notebook is mostly CPU-bound logic.
# ================================================================
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBEDDER_MODEL, device=device)

log.info("Loaded embedder %s on %s", EMBEDDER_MODEL, device)
print(f"\u2713 Embedder loaded: {EMBEDDER_MODEL} on {device}")

# ================================================================
# MODULE 2: Dense search (cosine similarity via matmul)
# ================================================================

def dense_search(
    query: str,
    embedder: SentenceTransformer,
    chunk_embeddings: np.ndarray,
    chunk_registry: list,
    top_k: int = 20,
) -> list:
    """
    Embed the query with bge-m3 and find the top-k most similar chunks
    by cosine similarity (dot product, since both sides are L2-normalized).

    Returns:
        list of (chunk_dict, cosine_score) tuples, sorted descending by score.
    """
    # Embed query (L2-normalized)
    q_emb = embedder.encode(
        query,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    # Cosine similarity = dot product (both sides normalized)
    scores = chunk_embeddings @ q_emb

    # Get top-k indices (argsort ascending, then reverse)
    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = []
    for idx in top_indices:
        results.append((chunk_registry[idx], float(scores[idx])))

    return results


# --- Quick smoke test ---
test_results = dense_search("eligibility criteria for admission", embedder, chunk_embeddings, chunk_registry, top_k=3)
print("Dense search smoke test (top 3):")
for chunk, score in test_results:
    print(f"  [{score:.4f}] {chunk['chunk_id']}  —  {chunk['chunk_text'][:80]}...")

# ================================================================
# MODULE 3: BM25 retrieval (lexical search)
# ================================================================
from rank_bm25 import BM25Okapi


def build_bm25_index(chunk_registry: list) -> BM25Okapi:
    """
    Tokenize chunk_text (lowercase + whitespace split) and build
    a BM25Okapi index. Cheap at this corpus size — build once, reuse.
    """
    tokenized_corpus = [
        c["chunk_text"].lower().split()
        for c in chunk_registry
    ]
    return BM25Okapi(tokenized_corpus)


def bm25_search(
    query: str,
    bm25_index: BM25Okapi,
    chunk_registry: list,
    top_k: int = 20,
) -> list:
    """
    Score all chunks with BM25 and return the top-k.

    Returns:
        list of (chunk_dict, bm25_score) tuples, sorted descending by score.
    """
    query_tokens = query.lower().split()
    scores = bm25_index.get_scores(query_tokens)

    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:  # skip zero-score results
            results.append((chunk_registry[idx], float(scores[idx])))

    return results


# --- Build index (once) ---
bm25_index = build_bm25_index(chunk_registry)
print(f"\u2713 BM25 index built over {len(chunk_registry)} chunks")

# --- Quick smoke test ---
test_results = bm25_search("eligibility criteria for admission", bm25_index, chunk_registry, top_k=3)
print("\nBM25 search smoke test (top 3):")
for chunk, score in test_results:
    print(f"  [{score:.4f}] {chunk['chunk_id']}  —  {chunk['chunk_text'][:80]}...")

# ================================================================
# MODULE 4: Reciprocal Rank Fusion
# ================================================================

def rrf_fuse(
    ranked_lists: list,
    k: int = 60,
    top_k: int = 20,
) -> list:
    """
    Standard Reciprocal Rank Fusion. Combines N ranked lists into one,
    using rank position (not raw score) so BM25/cosine scale mismatch
    doesn't matter.

    score(chunk) = sum over lists of  1 / (k + rank_in_that_list + 1)

    Args:
        ranked_lists: list of lists, each inner list is [(chunk_dict, score), ...]
                      sorted descending by score.
        k:            RRF constant (default 60, standard value).
        top_k:        Number of results to return.

    Returns:
        list of (chunk_dict, rrf_score) tuples, sorted descending by rrf_score.
    """
    rrf_scores = {}   # chunk_id -> cumulative rrf score
    chunk_map = {}    # chunk_id -> chunk_dict (for dedup)

    for ranked_list in ranked_lists:
        for rank, (chunk, _score) in enumerate(ranked_list):
            cid = chunk["chunk_id"]
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
            if cid not in chunk_map:
                chunk_map[cid] = chunk

    # Sort by RRF score descending
    sorted_ids = sorted(rrf_scores.keys(), key=lambda cid: rrf_scores[cid], reverse=True)

    results = []
    for cid in sorted_ids[:top_k]:
        results.append((chunk_map[cid], rrf_scores[cid]))

    return results


# --- Quick smoke test: fuse BM25 + Dense ---
test_q = "eligibility criteria for admission"
dense_res = dense_search(test_q, embedder, chunk_embeddings, chunk_registry, top_k=10)
bm25_res = bm25_search(test_q, bm25_index, chunk_registry, top_k=10)
fused = rrf_fuse([dense_res, bm25_res], top_k=5)

print("RRF fusion smoke test (BM25 + Dense, top 5):")
for chunk, score in fused:
    print(f"  [{score:.6f}] {chunk['chunk_id']}")

# ================================================================
# MODULE 5a: Query entity extraction (lexical, no LLM)
#
# Fuzzy/substring match query tokens and n-grams against
# name_index / alias_index keys to find candidate seed node_ids.
#
# Start with this lexical-match version — it's free and fast.
# An LLM-based query entity extractor can be swapped in later
# if recall is too low.
# ================================================================
import re

# Stopwords to skip when generating n-grams (same set as RAG MAIN)
STOPWORDS = {"the", "of", "and", "a", "an", "for", "in", "to", "on", "at",
             "is", "are", "was", "were", "be", "been", "being",
             "what", "which", "who", "whom", "how", "when", "where", "why",
             "do", "does", "did", "can", "could", "will", "would", "shall",
             "should", "may", "might", "must", "has", "have", "had",
             "this", "that", "these", "those", "it", "its",
             "i", "me", "my", "we", "our", "you", "your", "they", "their"}


def _normalize(text: str) -> str:
    """Normalize text for matching: lowercase, strip, collapse whitespace."""
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def _generate_ngrams(tokens: list, max_n: int = 5) -> list:
    """
    Generate all n-grams (n=1..max_n) from the token list.
    Returns list of (ngram_string, start_idx, end_idx) tuples.
    """
    ngrams = []
    for n in range(1, min(max_n + 1, len(tokens) + 1)):
        for i in range(len(tokens) - n + 1):
            gram = " ".join(tokens[i : i + n])
            ngrams.append((gram, i, i + n))
    return ngrams


def extract_query_entities(
    query: str,
    name_index: dict,
    alias_index: dict,
) -> list:
    """
    Extract seed entity node_ids from a query using lexical matching.

    Strategy:
    1. Generate n-grams (1-5 tokens) from the query.
    2. Check exact match against name_index and alias_index keys.
    3. Check substring containment in index keys for longer queries.
    4. Prefer longer n-gram matches (more specific).

    Returns:
        list of matched node_ids (deduplicated).
    """
    query_norm = _normalize(query)
    tokens = [t for t in query_norm.split() if t not in STOPWORDS]

    if not tokens:
        tokens = query_norm.split()  # fall back to all tokens

    matched = {}  # node_id -> match_length (longer = better)

    # --- Strategy 1: Exact n-gram match against indices ---
    ngrams = _generate_ngrams(tokens, max_n=5)
    # Sort by length descending so longer matches take priority
    ngrams.sort(key=lambda x: len(x[0]), reverse=True)

    for gram, _start, _end in ngrams:
        # Check name_index (canonical names)
        if gram in name_index:
            nid = name_index[gram]
            matched[nid] = max(matched.get(nid, 0), len(gram))

        # Check alias_index
        if gram in alias_index:
            nid = alias_index[gram]
            matched[nid] = max(matched.get(nid, 0), len(gram))

    # --- Strategy 2: Substring containment ---
    # For each index key, check if it appears as a substring in the query
    # (catches cases where the index key is a multi-word phrase)
    if len(matched) < 3:  # only if we haven't found many matches yet
        for key, nid in name_index.items():
            if len(key) >= 4 and key in query_norm:
                matched[nid] = max(matched.get(nid, 0), len(key))

        for key, nid in alias_index.items():
            if len(key) >= 4 and key in query_norm:
                matched[nid] = max(matched.get(nid, 0), len(key))

    # Sort by match length descending (more specific matches first)
    sorted_matches = sorted(matched.keys(), key=lambda nid: matched[nid], reverse=True)

    return sorted_matches


# --- Quick smoke test ---
test_entities = extract_query_entities("eligibility criteria for admission", name_index, alias_index)
print(f"Entity extraction smoke test: found {len(test_entities)} seed entities")
for nid in test_entities[:5]:
    ndata = G.nodes[nid]
    print(f"  {nid}  (type={ndata.get('type')}, name={ndata.get('name')})")

# ================================================================
# MODULE 5b: Graph BFS traversal
#
# BFS outward from seed nodes, up to n_hops, following edges in
# BOTH directions (treat graph as undirected for traversal —
# relations like PREREQUISITE_OF still connect relevant concepts
# regardless of direction).
# ================================================================
from collections import deque


def traverse_graph(
    seed_node_ids: list,
    G: nx.MultiDiGraph,
    n_hops: int = 2,
) -> dict:
    """
    BFS outward from seed nodes, up to n_hops, treating the directed
    graph as undirected (following edges in both directions).

    Returns:
        dict mapping node_id -> minimum hop distance from any seed.
        Includes seed nodes themselves (distance 0).
    """
    visited = {}  # node_id -> min hop distance
    queue = deque()  # (node_id, current_depth)

    # Initialize with seed nodes
    for seed in seed_node_ids:
        if seed in G:
            visited[seed] = 0
            queue.append((seed, 0))

    # BFS
    while queue:
        node, depth = queue.popleft()

        if depth >= n_hops:
            continue

        # Get neighbors in both directions (treat as undirected)
        neighbors = set()
        neighbors.update(G.successors(node))    # outgoing edges
        neighbors.update(G.predecessors(node))  # incoming edges

        for neighbor in neighbors:
            if neighbor not in visited:
                visited[neighbor] = depth + 1
                queue.append((neighbor, depth + 1))

    return visited


# --- Quick smoke test ---
if test_entities:
    visited = traverse_graph(test_entities[:2], G, n_hops=2)
    print(f"BFS smoke test: {len(test_entities[:2])} seeds -> {len(visited)} nodes visited")
    depth_counts = {}
    for nid, d in visited.items():
        depth_counts[d] = depth_counts.get(d, 0) + 1
    print(f"  By depth: {depth_counts}")
else:
    print("No test entities found — skipping BFS smoke test.")

# ================================================================
# MODULE 5c: Graph search — map visited nodes to chunks
#
# Orchestrates entity extraction → BFS → chunk collection → scoring.
# ================================================================

def graph_search(
    query: str,
    G: nx.MultiDiGraph,
    name_index: dict,
    alias_index: dict,
    chunk_registry: list,
    chunk_id_to_idx: dict,
    n_hops: int = 2,
    top_k: int = 20,
) -> list:
    """
    Graph-based retrieval pipeline:
    1. Extract seed entities from query (lexical matching).
    2. BFS traverse from seeds up to n_hops.
    3. Collect source_chunks from every visited node.
    4. Score each chunk by inverse hop distance from nearest seed
       (closer = higher score).
    5. Return (chunk_dict, score) tuples, deduped, sorted, top_k.

    If no seed entities are found, returns an empty list gracefully.
    """
    # Step 1: Extract seed entities
    seeds = extract_query_entities(query, name_index, alias_index)

    if not seeds:
        return []  # no graph signal for this query

    # Step 2: BFS traverse
    visited = traverse_graph(seeds, G, n_hops=n_hops)

    if not visited:
        return []

    # Step 3 & 4: Collect source_chunks and score by hop distance
    chunk_scores = {}  # chunk_id -> best (lowest hop distance) score

    for node_id, hop_dist in visited.items():
        node_data = G.nodes[node_id]
        source_chunks = node_data.get("source_chunks", [])

        # Score: higher for closer nodes (inverse hop distance)
        score = 1.0 / (1.0 + hop_dist)

        for cid in source_chunks:
            if cid in chunk_id_to_idx:  # verify chunk exists
                chunk_scores[cid] = max(chunk_scores.get(cid, 0.0), score)

    # Step 5: Sort by score, return top-k
    sorted_chunks = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for cid, score in sorted_chunks[:top_k]:
        idx = chunk_id_to_idx[cid]
        results.append((chunk_registry[idx], score))

    return results


# --- Quick smoke test ---
test_graph_res = graph_search(
    "eligibility criteria for admission",
    G, name_index, alias_index, chunk_registry, chunk_id_to_idx,
    n_hops=2, top_k=5
)
print(f"Graph search smoke test: {len(test_graph_res)} results")
for chunk, score in test_graph_res:
    print(f"  [{score:.4f}] {chunk['chunk_id']}  —  {chunk['chunk_text'][:80]}...")

# ================================================================
# MODULE 6: Final fusion — the single public entry point
#
# This is the function that Phase 4 (generation) and Phase 5
# (evaluation) will call.
# ================================================================

def retrieve(
    query: str,
    top_k: int = 10,
    n_hops: int = 2,
) -> list:
    """
    Hybrid retrieval: BM25 + Dense + Graph, fused via RRF.

    Args:
        query:  The user's question or search string.
        top_k:  Number of final chunks to return.
        n_hops: Number of hops for graph traversal.

    Returns:
        list of dicts — the top-k chunk_registry entries, ranked by
        fused relevance. Each dict has all chunk metadata plus an
        added 'retrieval_score' field.
    """
    # 1. BM25 lexical search
    bm25_results = bm25_search(query, bm25_index, chunk_registry, top_k=20)

    # 2. Dense semantic search
    dense_results = dense_search(query, embedder, chunk_embeddings, chunk_registry, top_k=20)

    # 3. Graph traversal search
    graph_results = graph_search(
        query, G, name_index, alias_index,
        chunk_registry, chunk_id_to_idx,
        n_hops=n_hops, top_k=20,
    )

    # 4. RRF fusion of all three
    fused = rrf_fuse([bm25_results, dense_results, graph_results], top_k=top_k)

    # 5. Return chunk dicts with retrieval_score attached
    final = []
    for chunk, rrf_score in fused:
        result = dict(chunk)  # copy to avoid mutating registry
        result["retrieval_score"] = rrf_score
        final.append(result)

    return final


# --- Smoke test ---
print("=" * 70)
print("retrieve() smoke test")
print("=" * 70)
test_final = retrieve("eligibility criteria for admission", top_k=5)
for i, chunk in enumerate(test_final, 1):
    print(f"\n--- Result {i} (score={chunk['retrieval_score']:.6f}) ---")
    print(f"    ID:      {chunk['chunk_id']}")
    print(f"    Doc:     {chunk['document_name']}")
    print(f"    Section: {chunk['section_title']}")
    print(f"    Text:    {chunk['chunk_text'][:120]}...")

# ================================================================

# --- Smoke test ---
r = retrieve("what is the attendance policy", top_k=3)
assert len(r) > 0 and "chunk_text" in r[0]
print(f"\n✓ retrieve() smoke test passed — {len(r)} results")


### Module 3: Load Qwen2.5-7B-Instruct
Loads the model using BitsAndBytes 4-bit quantization and fixes left-padding for batched inference.


In [ ]:
import os
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

if 'model' not in dir() or model is None:
    log.info("Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_CACHE_DIR if os.path.exists(MODEL_CACHE_DIR) else MODEL_NAME)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_CACHE_DIR if os.path.exists(MODEL_CACHE_DIR) else MODEL_NAME,
        device_map="auto"
    )
    
    # 5-token warmup test
    inputs = tokenizer("Hello", return_tensors="pt").to("cuda")
    _ = model.generate(**inputs, max_new_tokens=5)
    
    mem_gb = torch.cuda.memory_allocated() / 1e9
    print(f"✓ Qwen2.5-7B-Instruct-AWQ loaded — {mem_gb:.1f} GB VRAM used")
else:
    print("✓ Model already loaded.")



### Module 4: Prompt Builder
Constructs the context-grounded prompt securely within the 3,500 token budget.


In [ ]:
def build_prompt(query: str, chunks: list, max_context_tokens: int) -> str:
    # Approx 4 chars per token
    budget_chars = max_context_tokens * 4
    current_chars = 0
    formatted_context = ""
    
    for i, chunk in enumerate(chunks, 1):
        doc = chunk.get('document_name', 'Unknown Document')
        sec = chunk.get('section_title', 'General')
        page = chunk.get('page_number', 'N/A')
        text = chunk.get('chunk_text', '')
        
        chunk_str = f"[Source {i}: {doc}, Section: {sec}, Page {page}]\n{text}\n\n"
        
        if current_chars + len(chunk_str) > budget_chars:
            break
            
        formatted_context += chunk_str
        current_chars += len(chunk_str)
        
    if not formatted_context.strip():
        formatted_context = "No relevant context found."
        
    messages = [
        {"role": "system", "content": "You are a TIET policy assistant. Answer questions about Thapar Institute of Engineering and Technology rules, regulations, and policies using ONLY the provided context. Be precise and cite source numbers (e.g. [Source 1]) inline when you use information from them. If the answer cannot be found in the context, say: 'I could not find this information in the provided policy documents.'"},
        {"role": "user", "content": f"Context:\n{formatted_context.strip()}\n\nQuestion: {query}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

# --- Unit Test ---
mock_chunks = [{"document_name": "Test.pdf", "section_title": "1.0", "page_number": 1, "chunk_text": "Students must attend 75% of classes."}]
mock_prompt = build_prompt("What is the attendance policy?", mock_chunks, MAX_CONTEXT_TOKENS)
print("--- PROMPT PREVIEW ---")
print(mock_prompt[:500] + "...\n")
print("✓ build_prompt() unit test passed.")



### Module 5: Generator Orchestrator
End-to-end pipeline: retrieve -> prompt -> generate -> return structured citations.


In [ ]:
def generate_answer(query: str, top_k: int = 10, n_hops: int = 2, max_context_tokens: int = MAX_CONTEXT_TOKENS, verbose: bool = False) -> dict:
    chunks = retrieve(query, top_k=top_k, n_hops=n_hops)
    if not chunks:
        return {
            "query": query,
            "answer": "I could not find this information in the provided policy documents.",
            "chunks_used": 0,
            "chunks_retrieved": 0,
            "prompt_tokens": 0,
            "sources": []
        }
        
    prompt_str = build_prompt(query, chunks, max_context_tokens)
    inputs = tokenizer(prompt_str, return_tensors="pt").to("cuda")
    prompt_tokens = inputs["input_ids"].shape[1]
    
    if verbose:
        log.info(f"Generated prompt with {prompt_tokens} tokens")
        
    torch.cuda.empty_cache()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            temperature=0.1,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
        
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Calculate how many chunks actually fit in the prompt
    # Very rough approx:
    budget_chars = max_context_tokens * 4
    current_chars = 0
    chunks_used = 0
    for chunk in chunks:
        doc = chunk.get('document_name', 'Unknown Document')
        sec = chunk.get('section_title', 'General')
        page = chunk.get('page_number', 'N/A')
        text = chunk.get('chunk_text', '')
        chunk_str = f"[Source 1: {doc}, Section: {sec}, Page {page}]\n{text}\n\n"
        if current_chars + len(chunk_str) > budget_chars:
            break
        current_chars += len(chunk_str)
        chunks_used += 1
        
    sources = []
    for i in range(chunks_used):
        c = chunks[i]
        sources.append({
            "rank": i + 1,
            "chunk_id": c["chunk_id"],
            "document_name": c.get("document_name", ""),
            "section_title": c.get("section_title", ""),
            "page_number": c.get("page_number", None),
            "retrieval_score": c.get("retrieval_score", 0.0)
        })
        
    return {
        "query": query,
        "answer": answer,
        "chunks_used": chunks_used,
        "chunks_retrieved": len(chunks),
        "prompt_tokens": prompt_tokens,
        "sources": sources
    }

print("✓ generate_answer() initialized.")



### Module 6: End-to-End Demo
Runs 5 diverse queries to visually test the generation capabilities.


In [ ]:
queries = [
    "What is the attendance policy?",
    "What are the eligibility criteria for admission into the B.E./B.Tech. programmes?",
    "What are the rules regarding fee refund on withdrawal?",
    "What is the procedure for re-evaluation of answer scripts?",
    "What are the regulations concerning anti-ragging?"
]

print("=" * 60)
for q in queries:
    res = generate_answer(q, top_k=10, n_hops=2, verbose=False)
    print(f"Q: {res['query']}")
    print("-" * 60)
    print(f"A: {res['answer']}")
    print("-" * 60)
    print(f"Sources used ({res['chunks_used']}/{res['chunks_retrieved']} chunks, {res['prompt_tokens']} tokens):")
    for s in res['sources']:
        print(f"  [{s['rank']}] {s['document_name']} › {s['section_title']} (p.{s['page_number']}) — score {s['retrieval_score']:.4f}")
    print("=" * 60)

print("✓ Module 6 End-to-End Demo completed.")



### Module 7: Sanity QA Test
Runs the full generation pipeline against a random sample from the QA dataset. Note: This is for visual sanity-checking only. Formal evaluation metrics (RAGAS/LLM-as-a-judge) belong in Phase 5.


In [ ]:
from collections import defaultdict
import random

if 'qa_dataset' not in dir() or not qa_dataset:
    print("⚠ No QA dataset loaded — skipping Sanity QA Test.")
else:
    hop_types = ["single_hop", "cross_chunk", "2_hop", "3_hop", "4_hop"]
    by_hop = defaultdict(list)
    for q in qa_dataset:
        ht = q.get("hop_type", "unknown")
        if ht in hop_types:
            by_hop[ht].append(q)

    random.seed(42)
    sampled = []
    for ht in hop_types:
        pool = by_hop.get(ht, [])
        n_sample = min(2, len(pool))
        if n_sample > 0:
            sampled.extend(random.sample(pool, n_sample))

    print(f"Sampled {len(sampled)} questions for Sanity QA Test.")
    print("=" * 60)
    
    for q in sampled:
        question = q["question"]
        ht = q.get("hop_type", "unknown")
        res = generate_answer(question, top_k=10, n_hops=2)
        
        print(f"[{ht}] Q: {question}")
        print(f"A: {res['answer'][:300]}..." if len(res['answer']) > 300 else f"A: {res['answer']}")
        sources_str = ", ".join([f"[{s['rank']}]" for s in res['sources']])
        print(f"Sources: {sources_str}")
        print("-" * 60)

print("✓ Module 7 Sanity QA Test completed.")

